# Session 2 (Wed Jul 22) — Deploy it for real, then measure it

Two halves today.

1. **Deploy** — the bot moves from a laptop to a public URL on NRP
2. **Evaluate** — questions graded by hand, and the number that proves RAG helps

**No Docker images today.** You already shipped a Streamlit app in Week 4 by putting `app.py`
in a ConfigMap — we use that same trick, scaled up. Your code goes in one ConfigMap, the NRP
docs go in another, and a **PVC** holds the vector index so it survives a restart.

The pod does the work on startup: install deps → build the index → serve Streamlit. After
that, users just ask questions.


## Setup

The deploy half needs `kubectl` pointed at the cluster. The evaluation half rebuilds the
Session 1 RAG pipeline and draws two charts:

```bash
pip install "openai==1.55.0" "httpx<0.28" python-dotenv chromadb matplotlib
```

In [ ]:
import os, json, subprocess, textwrap
from dotenv import load_dotenv

load_dotenv()
NAMESPACE = "mfsada"          # the namespace we deploy into

def sh(cmd):
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(out.stdout or out.stderr)
    return out.stdout

sh("kubectl get pods -n " + NAMESPACE)

---
## A. Shipping code without building anything

In Week 4 you mounted `app.py` from a ConfigMap and pip-installed on boot. That still works,
and it's what we're using — no registry, no `docker build`, no waiting on a push.

Today there are **two** Python files instead of one:

| File | Job |
|---|---|
| `ingest.py` | Runs **once at startup**: chunk the docs, embed them, save to the PVC |
| `app.py` | The Streamlit UI: search the index, ground the answer, show sources |

And two ConfigMaps instead of one: **your code**, and **the NRP documentation itself**.

The container command chains them:

```
pip install ... && python /app/ingest.py && streamlit run /app/app.py
```

Because `ingest.py` writes to the PVC and checks before rebuilding, the *first* boot pays the
embedding cost and every boot after that skips straight to serving.

### A1. `ingest.py` — build the index at startup

In [ ]:
%%writefile ingest.py
"""Build the vector index at pod startup. Idempotent: skips if already built."""
import os, re, glob, logging
logging.getLogger("chromadb.telemetry.product.posthog").setLevel(logging.CRITICAL)
import chromadb
from openai import OpenAI

DOCS, DB = "/docs", "/data/chroma_db"
client = OpenAI(api_key=os.environ["NRP_LLM_TOKEN"],
                base_url=os.environ.get("NRP_LLM_BASE_URL", "https://ellm.nrp-nautilus.io/v1"),
                timeout=120)
EMBED_MODEL = os.environ.get("EMBED_MODEL", "qwen3-embedding")

def embed_batch(texts):
    return [d.embedding for d in client.embeddings.create(model=EMBED_MODEL, input=texts).data]

def chunk_text(t, size=2000, overlap=200):
    out, i = [], 0
    while i < len(t):
        out.append(t[i:i+size]); i += size - overlap
    return out

chunks = []
for path in sorted(glob.glob(f"{DOCS}/*.md")):
    raw = open(path, encoding="utf-8", errors="ignore").read()
    if len(raw.strip()) < 200:
        continue
    m = re.search(r"^Source:\s*(\S+)", raw, re.M)
    url = m.group(1) if m else "https://nrp.ai/documentation/"
    title = os.path.basename(path).replace(".md", "").split("_")[-1].replace("-", " ").title()
    for n, piece in enumerate(chunk_text(raw)):
        chunks.append({"id": f"{os.path.basename(path)}__{n:03d}", "source_url": url,
                       "title": title, "text": piece})
print(f"[ingest] {len(chunks)} chunks from {DOCS}", flush=True)

coll = chromadb.PersistentClient(path=DB).get_or_create_collection("nrp_docs")
if coll.count() >= len(chunks) and len(chunks) > 0:
    print(f"[ingest] index already built ({coll.count()} chunks) — skipping", flush=True)
else:
    for i in range(0, len(chunks), 32):
        b = chunks[i:i+32]
        coll.upsert(ids=[c["id"] for c in b], documents=[c["text"] for c in b],
                    embeddings=embed_batch([c["text"] for c in b]),
                    metadatas=[{"source_url": c["source_url"], "title": c["title"]} for c in b])
        print(f"[ingest] indexed {min(i+32, len(chunks))}/{len(chunks)}", flush=True)
print(f"[ingest] done — collection size {coll.count()}", flush=True)

**The `if coll.count() >= len(chunks)` check is the important line.** Without it every pod
restart would re-embed all the docs — minutes of waiting and a pile of needless API calls.
With it, the PVC does its job and a restart is just a pip install away from serving.

`flush=True` matters too: without it your progress prints sit in a buffer and
`kubectl logs` shows nothing while you're anxiously waiting.

### A2. `app.py` — the Streamlit UI

In [ ]:
%%writefile app.py
import os, logging
logging.getLogger("chromadb.telemetry.product.posthog").setLevel(logging.CRITICAL)
import streamlit as st, chromadb
from openai import OpenAI

DB = "/data/chroma_db"
client = OpenAI(api_key=os.environ["NRP_LLM_TOKEN"],
                base_url=os.environ.get("NRP_LLM_BASE_URL", "https://ellm.nrp-nautilus.io/v1"),
                timeout=120)
CHAT_MODEL = os.environ.get("CHAT_MODEL", "gpt-oss")
EMBED_MODEL = os.environ.get("EMBED_MODEL", "qwen3-embedding")

@st.cache_resource
def get_collection():
    return chromadb.PersistentClient(path=DB).get_or_create_collection("nrp_docs")

coll = get_collection()

def embed(t):
    return client.embeddings.create(model=EMBED_MODEL, input=[t]).data[0].embedding

def search(q, k=5):
    r = coll.query(query_embeddings=[embed(q)], n_results=k)
    return [{"text": d, "source_url": m["source_url"], "title": m["title"]}
            for d, m in zip(r["documents"][0], r["metadatas"][0])]

def build_prompt(q, hits):
    ctx = "\n\n---\n\n".join(f"[Source: {h['title']} | {h['source_url']}]\n{h['text']}" for h in hits)
    return (f"You are a helpful assistant for users of the National Research Platform (NRP).\n"
            f"Answer the question using ONLY the documentation below.\n"
            f"If the documentation does not contain the answer, say so honestly — do not guess.\n"
            f"Cite the source of each fact you use.\n\nDOCUMENTATION:\n{ctx}\n\nQUESTION: {q}")

st.set_page_config(page_title="NRP Docs Bot", page_icon="🤖")
st.title("🤖 NRP Docs Bot")
st.caption(f"Retrieval-augmented over {coll.count()} chunks of NRP documentation")

if "messages" not in st.session_state:
    st.session_state.messages = []
for m in st.session_state.messages:
    st.chat_message(m["role"]).write(m["content"])

if prompt := st.chat_input("Ask about NRP..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    st.chat_message("user").write(prompt)
    with st.chat_message("assistant"):
        with st.spinner("Searching the docs..."):
            hits = search(prompt)
            answer = client.chat.completions.create(
                model=CHAT_MODEL,
                messages=[{"role": "user", "content": build_prompt(prompt, hits)}]
            ).choices[0].message.content or "(no answer returned)"
        st.write(answer)
        with st.expander("Sources"):
            for h in hits:
                st.markdown(f"- [{h['title']}]({h['source_url']})")
    st.session_state.messages.append({"role": "assistant", "content": answer})

---
## B. Everything the pod needs, as cluster objects

No image means the pod starts empty and gets *everything* handed to it:

| Object | Holds | Why not somewhere else |
|---|---|---|
| **Secret** `chatbot-secrets` | the LLM token | never in code, never in a ConfigMap |
| **ConfigMap** `chatbot-code` | `app.py`, `ingest.py` | mounted at `/app` |
| **ConfigMap** `nrp-docs` | the 85 NRP doc pages | mounted at `/docs` |
| **PVC** `chatbot-data` | the vector index | mounted at `/data`, survives restarts |

In [ ]:
# ConfigMap keys can't contain "/", so flatten nrp-docs/a/b.md -> a_b.md
import shutil, pathlib
flat = pathlib.Path("nrp-docs-flat")
shutil.rmtree(flat, ignore_errors=True); flat.mkdir()
for src in pathlib.Path("nrp-docs").rglob("*.md"):
    shutil.copy(src, flat / str(src.relative_to("nrp-docs")).replace("/", "_"))
print("flattened", len(list(flat.glob("*.md"))), "docs ->", flat)

# The token — a Secret, never a ConfigMap.
sh(f"""kubectl create secret generic chatbot-secrets \\
  --from-literal=NRP_LLM_TOKEN='{os.environ["NRP_LLM_TOKEN"]}' \\
  -n {NAMESPACE} --dry-run=client -o yaml | kubectl apply -f -""")

# Your two Python files — small, so the idempotent apply pattern is fine.
sh(f"""kubectl create configmap chatbot-code \\
  --from-file=app.py --from-file=ingest.py \\
  -n {NAMESPACE} --dry-run=client -o yaml | kubectl apply -f -""")

# The docs are ~470 KB — too big for `kubectl apply` (see the note below).
sh(f"kubectl delete configmap nrp-docs -n {NAMESPACE} --ignore-not-found")
sh(f"kubectl create configmap nrp-docs --from-file={flat} -n {NAMESPACE}")

sh(f"kubectl get configmap -n {NAMESPACE}")

> **The `--dry-run=client -o yaml | kubectl apply -f -` pattern** makes a command
> **idempotent** — safe to run twice. Plain `kubectl create` fails the second time.

### ⚠️ Why the docs ConfigMap can't use `apply`

`kubectl apply` stores a full copy of the object in an annotation
(`last-applied-configuration`) so it can diff against it next time. That annotation is capped
at **256 KB** — and our docs are ~470 KB. You get:

```
metadata.annotations: Too long: may not be more than 262144 bytes
```

The data itself is fine (ConfigMaps allow ~1 MiB); it's the *annotation* that overflows. So
for big ConfigMaps: **delete, then create** — no annotation, no limit.

This is a genuinely confusing error the first time you meet it, because the object you're
applying is well under the ConfigMap limit.

**Never `kubectl get secret -o yaml` on a shared screen** — it prints the token in base64,
which is encoding, not encryption. Anyone watching can decode it.

---
## C. The PVC — why the vector database has to persist

A pod's filesystem is scratch space. Delete the pod and it's gone. Embedding thousands of
chunks takes minutes and costs API calls, so re-doing it on every restart would be painful
and slow every restart to a crawl.

A **PersistentVolumeClaim** is disk that outlives pods. Chroma writes there, and a restarted
pod picks up the existing index instantly.

In [ ]:
pvc = f"""apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: chatbot-data
  namespace: {NAMESPACE}
spec:
  accessModes: [ReadWriteOnce]
  resources:
    requests:
      storage: 10Gi
"""
open("pvc.yaml", "w").write(pvc)
sh("kubectl apply -f pvc.yaml")
sh(f"kubectl get pvc -n {NAMESPACE}")

---
## D. The Deployment

A **stock `python:3.12-slim` image** — no build, nothing pushed. The container installs
dependencies, builds the index, then serves. Read every block before we apply it; Student 5
has to explain this at the showcase.

In [ ]:
deployment = f"""apiVersion: apps/v1
kind: Deployment
metadata:
  name: chatbot
  namespace: {NAMESPACE}
  labels: {{app: chatbot}}
spec:
  replicas: 1          # <-- MUST stay 1. See the warning below.
  selector:
    matchLabels: {{app: chatbot}}
  template:
    metadata:
      labels: {{app: chatbot}}
    spec:
      containers:
      - name: streamlit
        image: python:3.12-slim          # stock image - nothing to build
        command: ["/bin/sh", "-c"]
        args:
          - >
            pip install --quiet --no-cache-dir streamlit openai chromadb &&
            python /app/ingest.py &&
            streamlit run /app/app.py
            --server.port=8501 --server.address=0.0.0.0 --server.headless=true
        env:
        - name: NRP_LLM_TOKEN
          valueFrom:
            secretKeyRef: {{name: chatbot-secrets, key: NRP_LLM_TOKEN}}
        - {{name: NRP_LLM_BASE_URL, value: "https://ellm.nrp-nautilus.io/v1"}}
        - {{name: CHAT_MODEL,  value: "gpt-oss"}}
        - {{name: EMBED_MODEL, value: "qwen3-embedding"}}
        ports:
        - containerPort: 8501
        readinessProbe:                  # first boot = pip install + embedding
          tcpSocket: {{port: 8501}}
          initialDelaySeconds: 30
          periodSeconds: 10
          failureThreshold: 60           # ~10 min of grace before it gives up
        volumeMounts:
        - {{name: code, mountPath: /app}}     # <- ConfigMap appears here as files
        - {{name: docs, mountPath: /docs}}    # <- the NRP documentation
        - {{name: data, mountPath: /data}}    # <- the PVC: index lives here
        resources:
          requests: {{cpu: "500m", memory: "2Gi"}}
          limits:   {{cpu: "2",    memory: "4Gi"}}
      volumes:
      - {{name: code, configMap: {{name: chatbot-code}}}}
      - {{name: docs, configMap: {{name: nrp-docs}}}}
      - {{name: data, persistentVolumeClaim: {{claimName: chatbot-data}}}}
"""
open("deployment.yaml", "w").write(deployment)
print(deployment)

### ⚠️ `replicas: 1` — leave it alone

Streamlit keeps a **websocket** open per user, and session state lives in the memory of the
pod that served the first request. With two replicas the load balancer sends your next
message to the other pod, which has never heard of your session.

What makes this genuinely nasty: **the logs stay perfectly clean.** Both pods report healthy,
nothing errors. The app just misbehaves in the browser — messages vanish, state resets. You
can lose a day to it. We have.

### Why a Deployment and not a bare Pod

A Pod that dies stays dead. A Deployment notices and starts a replacement. That's the whole
difference, and it's why nobody runs bare Pods in production.

---
## E. Service and Ingress — getting a public URL

- **Service** — a stable internal address. Pods come and go with changing IPs; the Service
  name doesn't change.
- **Ingress** — the public front door: hostname, HTTPS certificate, routing inward.

In [ ]:
INGRESS_HOST = "rehs-mfsada.nrp-nautilus.io"

networking = f"""apiVersion: v1
kind: Service
metadata:
  name: chatbot-svc
  namespace: {NAMESPACE}
spec:
  selector: {{app: chatbot}}
  ports:
  - port: 80
    targetPort: 8501
---
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: chatbot-ingress
  namespace: {NAMESPACE}
  annotations:
    nginx.ingress.kubernetes.io/proxy-read-timeout: "3600"   # websockets
spec:
  ingressClassName: haproxy
  tls:
  - hosts: [{INGRESS_HOST}]
    secretName: chatbot-tls
  rules:
  - host: {INGRESS_HOST}
    http:
      paths:
      - path: /
        pathType: Prefix
        backend:
          service:
            name: chatbot-svc
            port: {{number: 80}}
"""
open("networking.yaml", "w").write(networking)
print(networking)

That `proxy-read-timeout` annotation is not decoration. Streamlit's websocket is long-lived;
with the default timeout the proxy cuts it and the app reconnects every minute — which looks
exactly like a flaky app and is maddening to diagnose.

---
## F. Deploy it

The moment of truth.

In [ ]:
sh("kubectl apply -f deployment.yaml")
sh("kubectl apply -f networking.yaml")
sh(f"kubectl rollout status deploy/chatbot -n {NAMESPACE} --timeout=300s")

In [ ]:
sh(f"kubectl get pods,svc,ingress -n {NAMESPACE}")

If the pod isn't `Running`, the answer is almost always in one of these two:

```bash
kubectl describe pod <pod> -n mfsada   # scroll to Events at the bottom
kubectl logs <pod> -n mfsada
```

`ImagePullBackOff` → wrong image name or the registry needs credentials.
`CrashLoopBackOff` → the container starts and dies; the logs say why.
`Pending` → the cluster can't schedule it; `describe` explains what it's waiting for.

**Everyone open the URL on your phone, off wifi.** That's the project becoming real.

### F1. Prove the PVC works

Delete the pod. Kubernetes replaces it. Watch the logs: `ingest.py` should find the index
already built and **skip** re-embedding — that's the PVC earning its keep.

Expect roughly:

```
[ingest] 305 chunks from /docs
[ingest] index already built (305 chunks) - skipping
[ingest] done - collection size 305
```

The replacement still takes ~2 minutes, but that's the `pip install`, not re-embedding.
**That is the honest cost of not building an image** — and a fair thing to say at the
showcase if someone asks why you didn't use a registry.

In [ ]:
sh(f"kubectl delete pod -l app=chatbot -n {NAMESPACE}")
sh(f"kubectl rollout status deploy/chatbot -n {NAMESPACE} --timeout=600s")
sh(f"kubectl logs -l app=chatbot -n {NAMESPACE} --tail=-1 | grep ingest")
print("Skipped re-embedding? Then the PVC did its job.")

---
## G. Evaluation — does RAG actually help?

Here's the science half. We have a bot. Does retrieval *measurably* improve it, or does it
just feel better?

Student 6's questions get answered **twice** — once with retrieval, once without — and
graded by hand. The gap between those scores is the headline result of the whole project.

To score it we rebuild the exact Session 1 pipeline right here (in the merged repo this is
just `from src.ui.chat import answer_question`) so this half runs on its own, against the
same bundled NRP docs.

### G0. Rebuild the RAG pipeline

In [ ]:
import logging
logging.getLogger("chromadb.telemetry.product.posthog").setLevel(logging.CRITICAL)  # quiet noise
import glob, chromadb
from openai import OpenAI

client = OpenAI(api_key=os.environ["NRP_LLM_TOKEN"],
                base_url=os.environ.get("NRP_LLM_BASE_URL", "https://ellm.nrp-nautilus.io/v1"))
CHAT_MODEL, EMBED_MODEL = "gpt-oss", "qwen3-embedding"

def ask(prompt, model=CHAT_MODEL):
    return client.chat.completions.create(
        model=model, messages=[{"role": "user", "content": prompt}]).choices[0].message.content

def embed(text):
    return client.embeddings.create(model=EMBED_MODEL, input=[text]).data[0].embedding

def embed_batch(texts):
    return [d.embedding for d in client.embeddings.create(model=EMBED_MODEL, input=texts).data]

def chunk_text(text, size=2000, overlap=200):
    out, i = [], 0
    while i < len(text):
        out.append(text[i:i + size]); i += size - overlap
    return out

chunks = []
for path in glob.glob("nrp-docs/**/*.md", recursive=True):
    raw = open(path, encoding="utf-8", errors="ignore").read()
    if len(raw.strip()) < 200:
        continue
    rel = os.path.relpath(path, "nrp-docs").replace(".md", "")
    for n, piece in enumerate(chunk_text(raw)):
        chunks.append({"id": f"{rel.replace('/', '_')}__{n:03d}",
                       "source_url": f"https://nrp.ai/documentation/{rel}/",
                       "title": rel.split("/")[-1].replace("-", " ").title(), "text": piece})

coll = chromadb.PersistentClient(path="./chroma_db").get_or_create_collection("nrp_docs")
if coll.count() < len(chunks):                    # build once; the persisted index is reused after
    for i in range(0, len(chunks), 32):
        b = chunks[i:i + 32]
        coll.upsert(ids=[c["id"] for c in b], documents=[c["text"] for c in b],
                    embeddings=embed_batch([c["text"] for c in b]),
                    metadatas=[{"source_url": c["source_url"], "title": c["title"]} for c in b])
print("index ready:", coll.count(), "chunks")

def search(query, k=5):
    res = coll.query(query_embeddings=[embed(query)], n_results=k)
    return [{"text": d, "source_url": m["source_url"], "title": m["title"], "score": s}
            for d, m, s in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])]

def build_prompt(query, hits):
    context = "\n\n---\n\n".join(
        f"[Source: {h['title']} | {h['source_url']}]\n{h['text']}" for h in hits)
    return (f"You are a helpful assistant for users of the National Research Platform (NRP).\n"
            f"Answer the question using ONLY the documentation below.\n"
            f"If the documentation does not contain the answer, say so honestly — do not guess.\n"
            f"Cite the source of each fact you use.\n\nDOCUMENTATION:\n{context}\n\nQUESTION: {query}")

def answer_question(query, k=5):
    hits = search(query, k=k)
    return {"answer": ask(build_prompt(query, hits)), "chunks": hits}

In [ ]:
questions = [json.loads(l) for l in open("eval-questions.jsonl") if l.strip()]
print("questions:", len(questions))
print(json.dumps(questions[0], indent=2))

### G2. Answer each one both ways

In [ ]:
os.makedirs("eval", exist_ok=True)

rows = []
for i, item in enumerate(questions, 1):
    with_rag = answer_question(item["q"])["answer"]
    without  = ask(item["q"])
    rows.append({"question": item["q"], "expected": item.get("ideal", ""),
                 "with_rag": with_rag, "without_rag": without})
    print(f"{i:2d}/{len(questions)} done")

json.dump(rows, open("eval/results_raw.json", "w"), indent=2)
print("saved -> eval/results_raw.json")

### G3. Grade them — by hand, together

No shortcuts here. A model grading itself is not evidence, and a human reading 40 answers is
the most honest instrument we have. **1** = correct and grounded, **0** = wrong, vague, or
invented.

Print them in pairs and score as a group. Disagreements are the interesting part — argue
them out, because that argument is what you'll be asked about at the showcase.

In [ ]:
for i, r in enumerate(rows, 1):
    print("=" * 70)
    print(f"Q{i}: {r['question']}")
    print(f"\nEXPECTED: {r['expected']}")
    print(f"\n--- WITHOUT retrieval ---\n{textwrap.shorten(r['without_rag'], 400)}")
    print(f"\n--- WITH retrieval ---\n{textwrap.shorten(r['with_rag'], 400)}")

In [ ]:
# Fill these in as a group: 1 = correct, 0 = not. ONE entry per question.
scores_with    = [1, 1, 1, 1, 1, 1, 1, 1, 0, 0]
scores_without = [1, 0, 1, 1, 0, 0, 1, 0, 0, 0]

n = len(questions)
scores_with, scores_without = scores_with[:n], scores_without[:n]   # match however many questions
w, wo = sum(scores_with), sum(scores_without)
print(f"WITH retrieval:    {w}/{n}  ({w/n:.0%})")
print(f"WITHOUT retrieval: {wo}/{n}  ({wo/n:.0%})")
print(f"improvement:       +{w-wo} questions")

### G4. The accuracy chart

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4.5))
bars = ax.bar(["Without retrieval\n(plain LLM)", "With retrieval\n(our RAG bot)"],
              [wo, w], color=["#B0BEC5", "#1565C0"], width=0.55)
ax.set_ylabel("Questions answered correctly", fontsize=12)
ax.set_ylim(0, n)
ax.set_title(f"NRP question accuracy (n = {n})", fontsize=13, weight="bold")
for b, val in zip(bars, [wo, w]):
    ax.text(b.get_x() + b.get_width()/2, val + 0.3, f"{val}/{n}",
            ha="center", fontsize=13, weight="bold")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("eval/accuracy.png", dpi=200)
plt.show()

### G5. Where it fails

For every question the bot got wrong **with** retrieval, decide which kind of failure it was:

- **Retrieval miss** — the right document never came back. Look at `result["chunks"]`.
- **Model miss** — the right document *was* there and the answer still went wrong.

They need completely different fixes (chunking/`k` vs. prompt), which is exactly why the
distinction is worth calling out.

In [ ]:
retrieval_misses = 2      # count them from the run above
model_misses     = 1

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.bar(["Retrieval miss\n(wrong docs found)", "Model miss\n(right docs, bad answer)"],
       [retrieval_misses, model_misses], color=["#EF6C00", "#6A1B9A"], width=0.55)
ax.set_ylabel("Number of failures", fontsize=12)
ax.set_title("What kind of mistakes does it make?", fontsize=13, weight="bold")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("eval/failures.png", dpi=200)
plt.show()

---
## H. 🔧 Wrap-up

Look back at what you did today:

- **Deployed** the bot with no Docker image — code and docs in ConfigMaps, the index on a
  PVC — and got a public HTTPS URL anyone can open.
- **Measured** it honestly: correct answers with retrieval vs. without, graded by hand, with
  the failures broken down by cause.

That's a real system and an honest number to back it up. Thursday (Session 3) we give the bot
**tools** — live `kubectl` calls — and polish it into one chatbot that does both RAG and
tool calling. 🚀